# 🎬 Violence Detection Demo
**Three-stream late-fusion pipeline: RGB (R3D-18) + Joint (2s-AGCN) + Bone (2s-AGCN)**

Run cells 1→2→3→4→5 in order. After Cell 2 restarts the runtime, run cells 3→4→5 again.

In [ ]:
# ── CELL 1 ── Mount Drive & Install Dependencies ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Install mmcv from saved wheel (fast)
WHEEL = '/content/drive/MyDrive/mmcv-2.1.0-cp312-cp312-linux_x86_64.whl'
if os.path.exists(WHEEL):
    os.system(f'pip install {WHEEL} -q')

os.system('pip install mmengine mmpose mmdet decord gradio -q')

# Clone MMAction2 if not present
if not os.path.isdir('/content/mmaction2'):
    os.system('git clone https://github.com/open-mmlab/mmaction2.git -q /content/mmaction2')
    os.system('pip install -e /content/mmaction2 -q')
    os.system(
        "sed -i 's/from .multimodal import \\*/# from .multimodal import */' "
        "/content/mmaction2/mmaction/models/__init__.py"
    )
    os.system(
        "sed -i 's/checkpoint = torch.load(filename, map_location=map_location)/"
        "checkpoint = torch.load(filename, map_location=map_location, weights_only=False)/' "
        "/usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py"
    )

print('✅ Installation complete')

In [ ]:
# ── CELL 2 ── Restart Runtime (run once after Cell 1) ───────────────────────
# After running this cell, continue from Cell 3
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── CELL 3 ── Imports & Model Paths ─────────────────────────────────────────
import sys, os, warnings
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.models.video import r3d_18
import torch.nn as nn
from decord import VideoReader, cpu

warnings.filterwarnings('ignore')
sys.path.insert(0, '/content/mmaction2')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Model checkpoint paths ───────────────────────────────────────────────────
BASE       = '/content/drive/MyDrive/violence-detection'
RGB_CKPT   = f'{BASE}/checkpoints/rgb_v2/best_model.pth'
JOINT_CKPT = f'{BASE}/checkpoints/pose/2s-agcn-joint/best_acc_top1_epoch_8.pth'
BONE_CKPT  = f'{BASE}/checkpoints/pose/2s-agcn-bone/best_acc_top1_epoch_13.pth'
JOINT_CFG  = f'{BASE}/configs/2s-agcn_rwf2000_joint.py'
BONE_CFG   = f'{BASE}/configs/2s-agcn_rwf2000_bone.py'

# ── Fusion weights (grid search optimal) ────────────────────────────────────
W_RGB   = 0.45
W_JOINT = 0.25
W_BONE  = 0.30

print('✅ Config ready')

In [ ]:
# ── CELL 4 ── Load All Three Models ─────────────────────────────────────────
from mmaction.apis import init_recognizer
from ultralytics import YOLO

# 1. RGB Model (R3D-18 v2)
rgb_model = r3d_18()
rgb_model.fc = nn.Linear(rgb_model.fc.in_features, 2)
rgb_model.load_state_dict(torch.load(RGB_CKPT, map_location=device, weights_only=False))
rgb_model = rgb_model.to(device).eval()
print('✅ RGB model loaded (R3D-18 v2)')

# 2. Joint Model (2s-AGCN)
joint_model = init_recognizer(JOINT_CFG, JOINT_CKPT, device=str(device))
joint_model.eval()
print('✅ Joint model loaded (2s-AGCN)')

# 3. Bone Model (2s-AGCN)
bone_model = init_recognizer(BONE_CFG, BONE_CKPT, device=str(device))
bone_model.eval()
print('✅ Bone model loaded (2s-AGCN)')

# 4. YOLO Pose
yolo_model = YOLO('yolo11n-pose.pt')
print('✅ YOLO11n-pose loaded')

# ── RGB transforms ───────────────────────────────────────────────────────────
rgb_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((128, 171)),
    T.CenterCrop(112),
    T.ToTensor(),
    T.Normalize(mean=[0.43216, 0.394666, 0.37645],
                std=[0.22803, 0.22145, 0.216989])
])

print('\n✅ All models ready!')

In [ ]:
# ── CELL 5 ── Inference Functions + Gradio UI ────────────────────────────────
import random
import gradio as gr
from mmaction.utils import register_all_modules
register_all_modules()

COCO_PARENTS = [0,0,0,1,2,0,0,5,6,7,8,5,6,11,12,13,14]

def extract_pose(video_path, target_frames=150):
    """Extract YOLO11n-pose keypoints, keep top-2 persons per frame."""
    vr = VideoReader(video_path, ctx=cpu(0))
    total = len(vr)
    indices = np.linspace(0, total - 1, target_frames, dtype=int)
    pose_data = np.zeros((target_frames, 2, 17, 3), dtype=np.float32)

    for t, idx in enumerate(indices):
        frame = vr[idx].asnumpy()
        results = yolo_model(frame, verbose=False)[0]
        if results.keypoints is None or len(results.keypoints) == 0:
            continue
        detections = []
        for kpt, box in zip(results.keypoints.data.cpu().numpy(),
                             results.boxes.data.cpu().numpy()):
            area = (box[2]-box[0]) * (box[3]-box[1])
            score = area * float(np.mean(kpt[:, 2]))
            detections.append((score, kpt))
        detections.sort(key=lambda x: x[0], reverse=True)
        for m, (_, kpt) in enumerate(detections[:2]):
            pose_data[t, m] = kpt[:17]
    return pose_data

def compute_bone(joint_data):
    """Convert joint coordinates to directed bone vectors."""
    bone = np.zeros_like(joint_data)
    bone[..., 2] = joint_data[..., 2]
    for child, parent in enumerate(COCO_PARENTS):
        if child != 0:
            bone[:, :, child, :2] = joint_data[:, :, child, :2] - joint_data[:, :, parent, :2]
    return bone

def prepare_skeleton_input(pose_data, use_bone=False):
    """Prepare skeleton tensor for MMAction2."""
    data = compute_bone(pose_data) if use_bone else pose_data
    # (T, M, V, C) -> (M, T, V, C)
    kp = np.transpose(data[..., :2], (1, 0, 2, 3))   # (M, T, 17, 2)
    conf = np.transpose(data[..., 2], (1, 0, 2))      # (M, T, 17)
    return {
        'keypoint': torch.tensor(kp[None], dtype=torch.float32).to(device),
        'keypoint_score': torch.tensor(conf[None], dtype=torch.float32).to(device),
        'label': torch.tensor([0]),
        'img_shape': (320, 240),
        'ori_shape': (320, 240),
        'num_clips': 1,
        'clip_len': 100,
    }

def get_rgb_probs(video_path, n_frames=16):
    """Get RGB stream softmax probabilities."""
    vr = VideoReader(video_path, ctx=cpu(0))
    total = len(vr)
    indices = sorted(random.sample(range(total), min(n_frames, total)))
    frames = [vr[i].asnumpy() for i in indices]
    while len(frames) < n_frames:
        frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
    tensors = [rgb_transform(f) for f in frames]
    clip = torch.stack(tensors).permute(1, 0, 2, 3).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = rgb_model(clip)
    return F.softmax(logits, dim=1).cpu().numpy()[0]

def get_skeleton_probs(model, skeleton_input):
    """Get skeleton stream softmax probabilities."""
    from mmengine.structures import InstanceData
    from mmaction.structures import ActionDataSample
    data_sample = ActionDataSample()
    data_sample.gt_label = skeleton_input['label']
    batch = [{
        'inputs': skeleton_input['keypoint'],
        'data_samples': [data_sample],
    }]
    with torch.no_grad():
        result = model.test_step(batch)
    scores = result[0].pred_score.cpu().numpy()
    return scores

def predict(video_path):
    """Full three-stream late fusion prediction."""
    try:
        # Step 1: RGB stream
        rgb_probs = get_rgb_probs(video_path)

        # Step 2: Pose extraction
        pose_data = extract_pose(video_path)

        # Step 3: Joint stream
        joint_input = prepare_skeleton_input(pose_data, use_bone=False)
        joint_probs = get_skeleton_probs(joint_model, joint_input)

        # Step 4: Bone stream
        bone_input = prepare_skeleton_input(pose_data, use_bone=True)
        bone_probs = get_skeleton_probs(bone_model, bone_input)

        # Step 5: Late fusion
        fused = W_RGB * rgb_probs + W_JOINT * joint_probs + W_BONE * bone_probs

        label = '🚨 VIOLENCE DETECTED' if np.argmax(fused) == 1 else '✅ NON-VIOLENT'

        return {
            f'{label}': float(max(fused)),
            'Stream scores': f'RGB: {rgb_probs[1]:.2f} | Joint: {joint_probs[1]:.2f} | Bone: {bone_probs[1]:.2f}'
        }

    except Exception as e:
        return {'Error': str(e)}

# ── Gradio Interface ─────────────────────────────────────────────────────────
demo = gr.Interface(
    fn=predict,
    inputs=gr.Video(label='Upload surveillance video (.mp4 / .avi)'),
    outputs=gr.JSON(label='Prediction'),
    title='🎬 Violence Detection System',
    description=(
        'Three-stream late-fusion pipeline combining R3D-18 RGB stream '
        'with Joint and Bone 2s-AGCN streams.\n'
        f'Fusion weights: RGB={W_RGB} | Joint={W_JOINT} | Bone={W_BONE}'
    ),
    theme=gr.themes.Soft()
)

demo.launch(share=True, debug=False)